# The Fab-Line Game — G1: one bad knob → a dead die

The first banked artifact of the fab-line game (plan `docs/plans/fab-game.md` §§1, 6; ADR 0005).
It runs the **same wafer** through the *already-validated* diffusion → oxidation → lithography →
device back end — once **in focus**, once with a single bad knob (**defocus**) — and shows the
consequence ripple end to end:

> defocus the exposure → the aerial image loses edge sharpness (**NILS** collapses) → the gate no
> longer prints reliably → those dies leave the spec window → the wafer **yield** drops, and the
> **failure trail** names *defocus* as the cause.

A center-to-edge focus bowl makes the failure **spatial** — the edge ring dies while the centre
survives — and a litho **rework** (strip resist, re-expose at corrected focus) recovers it.
**All reuse, zero new physics:** every number traces to a `chip/` module that already passes its
triad. This notebook is a thin skin over the validated harness (ADR 0002/0005 — a figure is never
in the correctness path); the assertions live in `fab_game/tests/`.

In [ ]:
%matplotlib inline
from fab_game import run_line, wafer_yield, diagnose, DEFAULT_SPECS, Variation
from fab_game.recipe import Recipe, LithoKnobs
from fab_game.demo_fab_game import compute, print_summary

result = compute()
print_summary(result)

## The wafer maps

Green = a good die, red = a dead one. The good recipe yields a full wafer; one bad knob kills an
**edge ring**; the `NILS`-vs-radius panel shows *why* (the printability floor catches the
defocused edge dies); rework brings it back.

In [ ]:
from fab_game.plots import fab_game_figure

fig = fab_game_figure(result)
fig

## Turn the focus knob yourself

Sweep the **defocus** knob and watch the yield fall and the edge ring grow. (With `ipywidgets`
installed you get a live slider; otherwise the cell prints a static sweep so the notebook still
runs headless.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fab_game.plots import _wafer_map


def show_wafer(defocus_nm=90.0, seed=0):
    """Run the line at this defocus and draw the wafer map + the worst die's failure trail."""
    recipe = Recipe(litho=LithoKnobs(defocus_nm=float(defocus_nm)))
    wafer = run_line(recipe, seed=seed, variation=Variation(), specs=DEFAULT_SPECS, grid_n=7)
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    _wafer_map(ax, wafer, f"defocus = {defocus_nm:.0f} nm")
    plt.show()
    dead = [d for d in wafer.dies if d.verdict.failed]
    if dead:
        worst = max(dead, key=lambda d: d.radius_frac)
        print(diagnose(worst))
    else:
        print(f"yield {wafer_yield(wafer):.0%} — every die printed in spec.")


try:
    from ipywidgets import interact, FloatSlider
    interact(show_wafer, defocus_nm=FloatSlider(min=0, max=320, step=10, value=90,
                                                description="defocus (nm)"),
             seed=(0, 5))
except ImportError:
    for z in (0.0, 90.0, 200.0):
        print(f"\n--- defocus = {z:.0f} nm ---")
        show_wafer(z)

## G2 — down the boule: Scheil segregation → a V_t spread

The **first new front-of-line physics** (`chip.czochralski`, cited + triad-tested). A Czochralski
boule grown from a boron melt has a **Scheil** axial profile — `C_s(z) = N_seed·(1−z)^(k−1)` — so
because boron's segregation coefficient `k ≈ 0.8 < 1`, the solid that freezes later is *more* doped.
Slice a wafer batch down the boule and that rising substrate doping **alone** — one crystal-growth
knob, no process change — walks the device `V_t` up across the spec window, so the boule's **tail is
scrapped**. (Unit-of-run, plan §10: a *run* is one wafer at axial `slice_z`; `run_batch` is the
sweep that surfaces where each slice sits on the Scheil curve.)


In [ ]:
from fab_game.demo_boule import compute as compute_boule, print_summary as print_boule
from fab_game.plots import boule_figure

boule_result = compute_boule()
print_boule(boule_result)
boule_figure(boule_result)


## G3 — the die map made physical: killer particles → functional yield + the TTV scrap

New cited physics (`chip.wafer_prep`, triad-tested): the **defect-limited yield law**. Killer
particles are scattered at **locations** across the wafer; a die that catches one is dead
*functionally* (the transistor exists but is killed). The probability a die of critical area `A`
catches **zero** killers at density `D₀` is `Y = exp(−D₀·A)` — the **Murphy / Poisson** law.

The placement is a seeded **per-die Poisson** process; sweep the density and the empirical defect
yield **converges to the cited `exp(−D₀·A)` curve** — the random placement tied back to the
validated physics (same byte-identical die area on both sides). Geometry (TTV/bow) is a **wafer-level
scrap** gate: a weak CMP leaves the TTV out of flatness spec → the wafer is scrapped, recoverable by
a re-polish that **eats thickness**.


In [ ]:
from fab_game.demo_wafer_prep import compute as compute_prep, print_summary as print_prep
from fab_game.plots import wafer_prep_figure

prep_result = compute_prep()
print_prep(prep_result)
wafer_prep_figure(prep_result)


## Where this goes next

G1 proved the **mechanism** (state → propagation → spec → yield → rework); **G2** added the first
new cited physics (Czochralski + the Scheil axial-resistivity boule); **G3** (above) made the
across-wafer die map physical — killer particles → functional yield (the cited Poisson law) + the
TTV/bow geometry scrap. The line grows front-to-back from here (plan §6): **G4** adds silicon
purification + the contamination consequence model (segregation refining, and the metal-lifetime /
oxygen→thermal-donor stories deferred from G2); later phases add etch/deposition, packaging, and the
roguelike shell.
